In [ ]:
from tmkb.configuration import load_config, ConfigSchema
from tmkb.kb_modules import (
    OctaveBuilder,
    MxKeyBuilder,
    MxKeyType,
    CrontrollerCaseBuilder,
    EndCapBuilder,
    BackplateBuilder,
)
from tmkb.kb_modules.protocols import HasKeys
from tmkb.generate import _get_partial_keys_cnt
from tmkb.primitives.base_builders import RelativeCoords, XPos, YPos, ZPos, GroupBuilder

from pathlib import Path
from jupyterscad import view

In [ ]:
conf = load_config(Path("../configs/default.conf.yaml"))
conf

In [ ]:
def key_to_preview(builder: MxKeyBuilder):
    builder.move([conf.mount_u / 2, conf.mount_u / 2, conf.mount_u / 2])
    return builder.build(rotation=180)


def build_with_keys(builder: HasKeys, conf: ConfigSchema):
    model = builder.build(printing_orientation=False)
    for type_ in MxKeyType:
        for x, y, z in builder.get_total_key_positions(type_):
            key: MxKeyBuilder = MxKeyBuilder.mx_key_factory(type_, conf)
            key.move([x, y, z])
            model += key_to_preview(key)
    return model

In [ ]:
from enum import Flag


class PreviewOptions(Flag):
    KEYS = 1
    BACKPLATES = 2
    SEPARATORS = 4

In [ ]:
def get_positioned_backplate_model(
    builder: GroupBuilder, seperate: bool, conf: ConfigSchema
):
    backplate = BackplateBuilder(builder, conf)
    backplate.move_rel(
        builder,
        anchor=RelativeCoords(zpos=ZPos.BOTTOM),
        alignment=RelativeCoords(xpos=XPos.CENTER, ypos=YPos.CENTER),
    )
    if seperate:
        backplate.move([0, 0, -10])
    return backplate.build()


def generate_preview_model(options: PreviewOptions, conf: ConfigSchema):
    has_separators = PreviewOptions.SEPARATORS in options

    partial_octave_num_of_keys = _get_partial_keys_cnt(conf.key_count)
    keys_per_octave = [7] * (conf.key_count // 12) + [partial_octave_num_of_keys[0]]

    octaves = [OctaveBuilder(n, conf) for n in keys_per_octave]

    case_ = CrontrollerCaseBuilder(octaves[0], conf)
    cap = EndCapBuilder(octaves[0], conf)

    model = cap.build(printing_orientation=False)
    if PreviewOptions.BACKPLATES in options:
        model += get_positioned_backplate_model(cap, has_separators, conf)

    prev = cap
    for el in reversed([case_] + octaves):
        el.move_rel(
            prev,
            anchor=RelativeCoords(xpos=XPos.LEFT),
            alignment=RelativeCoords(ypos=YPos.CENTER),
        )
        if PreviewOptions.SEPARATORS in options:
            el.move([-10, 0, 0])

        if PreviewOptions.KEYS in options:
            model += build_with_keys(el, conf)
        else:
            model += el.build(printing_orientation=False)

        if PreviewOptions.BACKPLATES in options:
            model += get_positioned_backplate_model(el, has_separators, conf)

        prev = el
    return model


view(generate_preview_model(PreviewOptions.SEPARATORS, conf))